In [1]:
!pip show tensorflow

Name: tensorflow
Version: 2.19.0
Summary: TensorFlow is an open source machine learning framework for everyone.
Home-page: https://www.tensorflow.org/
Author: Google Inc.
Author-email: packages@tensorflow.org
License: Apache 2.0
Location: /home/stemland/.local/lib/python3.10/site-packages
Requires: absl-py, astunparse, flatbuffers, gast, google-pasta, grpcio, h5py, keras, libclang, ml-dtypes, numpy, opt-einsum, packaging, protobuf, requests, setuptools, six, tensorboard, tensorflow-io-gcs-filesystem, termcolor, typing-extensions, wrapt
Required-by: 


In [5]:
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
import numpy as np
import shutil

# Setup: Your base directory where train and test folders are
base_dir = os.path.expanduser('/home/stemland/dataset')

# Your data is already in these directories
# Ensure THESE directories contain 'cat' and 'dog' subfolders with images
train_dir = os.path.join(base_dir, 'train')
test_dir = os.path.join(base_dir, 'test')

# --- Optional: Add checks to ensure your train/test dirs exist and have subfolders ---
print(f"Checking if train directory exists: {os.path.exists(train_dir)}")
print(f"Checking if test directory exists: {os.path.exists(test_dir)}")
if os.path.exists(train_dir):
    print(f"Contents of train directory: {os.listdir(train_dir)}")
    # Add a check for files within the cat/dog subfolders
    train_cat_path = os.path.join(train_dir, 'cat')
    train_dog_path = os.path.join(train_dir, 'dog')
    if os.path.exists(train_cat_path):
        print(f"Files in {train_cat_path}: {len([f for f in os.listdir(train_cat_path) if os.path.isfile(os.path.join(train_cat_path, f))])}")
    if os.path.exists(train_dog_path):
         print(f"Files in {train_dog_path}: {len([f for f in os.listdir(train_dog_path) if os.path.isfile(os.path.join(train_dog_path, f))])}")

if os.path.exists(test_dir):
    print(f"Contents of test directory: {os.listdir(test_dir)}")
    # Add a check for files within the cat/dog subfolders
    test_cat_path = os.path.join(test_dir, 'cat')
    test_dog_path = os.path.join(test_dir, 'dog')
    if os.path.exists(test_cat_path):
        print(f"Files in {test_cat_path}: {len([f for f in os.listdir(test_cat_path) if os.path.isfile(os.path.join(test_cat_path, f))])}")
    if os.path.exists(test_dog_path):
         print(f"Files in {test_dog_path}: {len([f for f in os.listdir(test_dog_path) if os.path.isfile(os.path.join(test_dog_path, f))])}")
# ------------------------------------------------------------------------------------


# Load images directly from your existing train and test directories
train_datagen = ImageDataGenerator(rescale=1./255)
test_datagen  = ImageDataGenerator(rescale=1./255)

print("\nLoading training data with flow_from_directory...")
train_data = train_datagen.flow_from_directory(
    train_dir, # Point directly to your existing train directory
    target_size=(150, 150),
    batch_size=16,
    class_mode='binary'
)

print("\nLoading testing data with flow_from_directory...")
test_data = test_datagen.flow_from_directory(
    test_dir, # Point directly to your existing test directory
    target_size=(150, 150),
    batch_size=16,
    class_mode='binary'
)

# Check if the datasets are empty
print(f"\nImages found by flow_from_directory (training): {train_data.samples}")
print(f"Images found by flow_from_directory (testing): {test_data.samples}")


if train_data.samples == 0:
    print(f"Error: flow_from_directory found 0 training images in {train_dir}. Please check the directory structure and image formats.")
elif test_data.samples == 0:
    print(f"Error: flow_from_directory found 0 testing images in {test_dir}. Please check the directory structure and image formats.")
else:
    # Build model
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
        layers.MaxPooling2D(2, 2),

        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D(2, 2),

        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dense(1, activation='sigmoid')  # binary output
    ])

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    # Train model
    print("Starting model training...")
    model.fit(train_data, epochs=5, validation_data=test_data)

    # Save model (optional)
    model.save("cat_dog_model.h5")

Checking if train directory exists: True
Checking if test directory exists: True
Contents of train directory: ['dog', 'cat']
Files in /home/stemland/dataset/train/cat: 56
Files in /home/stemland/dataset/train/dog: 56
Contents of test directory: ['dog', 'cat']
Files in /home/stemland/dataset/test/cat: 14
Files in /home/stemland/dataset/test/dog: 14

Loading training data with flow_from_directory...
Found 112 images belonging to 2 classes.

Loading testing data with flow_from_directory...
Found 28 images belonging to 2 classes.

Images found by flow_from_directory (training): 112
Images found by flow_from_directory (testing): 28


/home/stemland/.local/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2025-05-18 09:23:14.866662: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Starting model training...
Epoch 1/5


/home/stemland/.local/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


7/7 ━━━━━━━━━━━━━━━━━━━━ 6s 366ms/step - accuracy: 0.4592 - loss: 1.8553 - val_accuracy: 0.3929 - val_loss: 0.7230
Epoch 2/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 239ms/step - accuracy: 0.7590 - loss: 0.6433 - val_accuracy: 0.3571 - val_loss: 0.7594
Epoch 3/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 212ms/step - accuracy: 0.6709 - loss: 0.5983 - val_accuracy: 0.4286 - val_loss: 0.8529
Epoch 4/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 216ms/step - accuracy: 0.7269 - loss: 0.5820 - val_accuracy: 0.3214 - val_loss: 0.8045
Epoch 5/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 211ms/step - accuracy: 0.9587 - loss: 0.4055 - val_accuracy: 0.3214 - val_loss: 0.9339


In [6]:
# This cell contains the data loading and model training.
# Ensure your images are in /home/stemland/dataset/train/cat, /home/stemland/dataset/train/dog, etc.
# Run this cell and let it complete all 5 epochs.

import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
import numpy as np
import shutil # Still needed for potential cleanup, but not for copying in this case

# Setup: Your base directory where train and test folders are
base_dir = os.path.expanduser('/home/stemland/dataset')

# Your data is already in these directories
# Ensure THESE directories contain 'cat' and 'dog' subfolders with images
train_dir = os.path.join(base_dir, 'train')
test_dir = os.path.join(base_dir, 'test')

# --- Optional: Add checks to ensure your train/test dirs exist and have subfolders ---
print(f"Checking if train directory exists: {os.path.exists(train_dir)}")
print(f"Checking if test directory exists: {os.path.exists(test_dir)}")
if os.path.exists(train_dir):
    print(f"Contents of train directory: {os.listdir(train_dir)}")
    # Add a check for files within the cat/dog subfolders
    train_cat_path = os.path.join(train_dir, 'cat')
    train_dog_path = os.path.join(train_dir, 'dog')
    if os.path.exists(train_cat_path):
        print(f"Files in {train_cat_path}: {len([f for f in os.listdir(train_cat_path) if os.path.isfile(os.path.join(train_cat_path, f))])}")
    if os.path.exists(train_dog_path):
         print(f"Files in {train_dog_path}: {len([f for f in os.listdir(train_dog_path) if os.path.isfile(os.path.join(train_dog_path, f))])}")

if os.path.exists(test_dir):
    print(f"Contents of test directory: {os.listdir(test_dir)}")
    # Add a check for files within the cat/dog subfolders
    test_cat_path = os.path.join(test_dir, 'cat')
    test_dog_path = os.path.join(test_dir, 'dog')
    if os.path.exists(test_cat_path):
        print(f"Files in {test_cat_path}: {len([f for f in os.listdir(test_cat_path) if os.path.isfile(os.path.join(test_cat_path, f))])}")
    if os.path.exists(test_dog_path):
         print(f"Files in {test_dog_path}: {len([f for f in os.listdir(test_dog_path) if os.path.isfile(os.path.join(test_dog_path, f))])}")
# ------------------------------------------------------------------------------------


# Load images directly from your existing train and test directories
train_datagen = ImageDataGenerator(rescale=1./255)
test_datagen  = ImageDataGenerator(rescale=1./255)

print("\nLoading training data with flow_from_directory...")
train_data = train_datagen.flow_from_directory(
    train_dir, # Point directly to your existing train directory
    target_size=(150, 150),
    batch_size=16,
    class_mode='binary'
)

print("\nLoading testing data with flow_from_directory...")
test_data = test_datagen.flow_from_directory(
    test_dir, # Point directly to your existing test directory
    target_size=(150, 150),
    batch_size=16,
    class_mode='binary'
)

# Check if the datasets are empty - This should now show non-zero values!
print(f"\nImages found by flow_from_directory (training): {train_data.samples}")
print(f"Images found by flow_from_directory (testing): {test_data.samples}")


if train_data.samples == 0:
    print(f"Error: flow_from_directory found 0 training images in {train_dir}. Please check the directory structure and image formats.")
elif test_data.samples == 0:
    print(f"Error: flow_from_directory found 0 testing images in {test_dir}. Please check the directory structure and image formats.")
else:
    # Build model
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
        layers.MaxPooling2D(2, 2),

        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D(2, 2),

        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dense(1, activation='sigmoid')  # binary output
    ])

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    # Train model
    print("Starting model training...")
    # Ensure this completes without interruption!
    model.fit(train_data, epochs=5, validation_data=test_data)

    # Save model (optional)
    model.save("cat_dog_model.h5")

# --- Separate cell for prediction after successful training and saving ---
# # %%
# import tensorflow as tf
# from tensorflow.keras.models import load_model
# from tensorflow.keras.preprocessing import image
# import numpy as np
# import os
#
# # Load the saved model
# # Make sure "cat_dog_model.h5" exists in the same directory as your notebook
# # or provide the full path if saved elsewhere.
# model = load_model("cat_dog_model.h5")
#
# # Function to predict on a single image (Same function as before)
# def predict_cat_or_dog(image_path, model, target_size=(150, 150)):
#     # ... (function body)
#     if not os.path.exists(image_path):
#         return f"Error: Image not found at {image_path}"
#     try:
#         img = image.load_img(image_path, target_size=target_size)
#         img_array = image.img_to_array(img)
#         img_array = np.expand_dims(img_array, axis=0)
#         img_array /= 255.
#         prediction = model.predict(img_array)
#         if prediction[0][0] > 0.5:
#             return f"Predicted: Dog ({prediction[0][0]:.4f})"
#         else:
#             return f"Predicted: Cat ({prediction[0][0]:.4f})"
#     except Exception as e:
#         return f"Error processing image {image_path}: {e}"
#
# # --- Example Usage (in the prediction cell) ---
# test_image_path_cat = '/path/to/a/new/cat/image.jpg' # <-- Change this path
# test_image_path_dog = '/path/to/a/new/dog/image.jpg' # <-- Change this path
# test_image_path_missing = '/this/path/does/not/exist.png'
#
# print(predict_cat_or_dog(test_image_path_cat, model))
# print(predict_cat_or_dog(test_image_path_dog, model))
# print(predict_cat_or_dog(test_image_path_missing, model))

Checking if train directory exists: True
Checking if test directory exists: True
Contents of train directory: ['dog', 'cat']
Files in /home/stemland/dataset/train/cat: 56
Files in /home/stemland/dataset/train/dog: 56
Contents of test directory: ['dog', 'cat']
Files in /home/stemland/dataset/test/cat: 14
Files in /home/stemland/dataset/test/dog: 14

Loading training data with flow_from_directory...
Found 112 images belonging to 2 classes.

Loading testing data with flow_from_directory...
Found 28 images belonging to 2 classes.

Images found by flow_from_directory (training): 112
Images found by flow_from_directory (testing): 28
Starting model training...
Epoch 1/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 276ms/step - accuracy: 0.4315 - loss: 1.8440 - val_accuracy: 0.5000 - val_loss: 0.7732
Epoch 2/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 211ms/step - accuracy: 0.5693 - loss: 0.6654 - val_accuracy: 0.5000 - val_loss: 0.7141
Epoch 3/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 210ms/step - accuracy: 0.8419 - loss: 0.5826 - val

In [7]:
# This is the cell for prediction.
# Run this AFTER the training cell has completed and saved the model.

import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import numpy as np
import os

# Load the saved model
# Make sure "cat_dog_model.h5" exists in the same directory as your notebook
# or provide the full path to the file.
model_path = "cat_dog_model.h5"

# Added check to be sure the model file exists before trying to load
if os.path.exists(model_path):
    print(f"Loading model from {model_path}...")
    model = load_model(model_path)

    # Function to predict on a single image (Same function as before)
    def predict_cat_or_dog(image_path, model, target_size=(150, 150)):
        """
        Loads an image, preprocesses it, and predicts if it's a cat or a dog.

        Args:
            image_path (str): The full path to the image file.
            model (tf.keras.Model): The trained Keras model.
            target_size (tuple): The size the image should be resized to (width, height).

        Returns:
            str: The predicted class ('cat' or 'dog') or an error message.
        """
        if not os.path.exists(image_path):
            return f"Error: Image not found at {image_path}"

        try:
            img = image.load_img(image_path, target_size=target_size)
            img_array = image.img_to_array(img)
            img_array = np.expand_dims(img_array, axis=0)
            img_array /= 255.
            prediction = model.predict(img_array)
            # Assuming 'cat' is 0 and 'dog' is 1 based on alphabetical order
            if prediction[0][0] > 0.5:
                return f"Predicted: Dog ({prediction[0][0]:.4f})"
            else:
                return f"Predicted: Cat ({prediction[0][0]:.4f})"
        except Exception as e:
            return f"Error processing image {image_path}: {e}"

    # --- Example Usage ---

    # !!! IMPORTANT: Replace these paths with actual paths to NEW images you want to test !!!
    # Example: You might have a new image at /home/stemland/new_images/test_cat_001.jpg
    test_image_path_cat = '/path/to/a/new/cat/image.jpg' # <-- Change this path
    test_image_path_dog = '/path/to/a/new/dog/image.jpg' # <-- Change this path
    test_image_path_missing = '/this/path/does/not/exist.png' # Example of a missing image

    print(predict_cat_or_dog(test_image_path_cat, model))
    print(predict_cat_or_dog(test_image_path_dog, model))
    print(predict_cat_or_dog(test_image_path_missing, model))

else:
    print(f"Error: Model file not found at {model_path}.")
    print("Please ensure the model training and saving step was completed successfully in the previous cell.")

# You can also check the class indices used during training (run this in the training cell if needed):
# print(train_data.class_indices)

Loading model from cat_dog_model.h5...


Error: Image not found at /path/to/a/new/cat/image.jpg
Error: Image not found at /path/to/a/new/dog/image.jpg
Error: Image not found at /this/path/does/not/exist.png


In [23]:
# --- Example Usage ---

# Replace with the path to an actual image you want to test
# Example: You might have a new image at /home/stemland/new_images/test_cat.jpg
# test_image_path_cat = '/path/to/a/new/cat/image.jpg' # <-- This is the placeholder you need to change
test_image_path_cat = '/home/stemland/dataset/test/cat/cat_5.jpg' # <--- REPLACE this with the ACTUAL full path to your cat image

# test_image_path_dog = '/path/to/a/new/dog/image.jpg' # <-- This is the placeholder you need to change
test_image_path_dog = '/home/stemland/dataset/test/dog/dog_68.jpg' # <--- REPLACE this with the ACTUAL full path to your dog image

# test_image_path_missing = '/this/path/does/not/exist.png' # This is just an example of a non-existent path

In [1]:
# This is the cell for prediction.
# Run this AFTER the training cell has completed and saved the model.

import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import numpy as np
import os

# Load the saved model
model_path = "cat_dog_model.h5"

# Added check to be sure the model file exists before trying to load
if os.path.exists(model_path):
    print(f"Loading model from {model_path}...")
    model = load_model(model_path)
    print("Model loaded successfully.") # <-- Added print

    # Function to predict on a single image (Same function as before)
    def predict_cat_or_dog(image_path, model, target_size=(150, 150)):
        """
        Loads an image, preprocesses it, and predicts if it's a cat or a dog.

        Args:
            image_path (str): The full path to the image file.
            model (tf.keras.Model): The trained Keras model.
            target_size (tuple): The size the image should be resized to (width, height).

        Returns:
            str: The predicted class ('cat' or 'dog') or an error message.
        """
        print(f"\nAttempting to predict on image: {image_path}") # <-- Added print
        if not os.path.exists(image_path):
            print(f"Debug: os.path.exists({image_path}) is False") # <-- Added print
            return f"Error: Image not found at {image_path}"

        try:
            print("Debug: Loading image...") # <-- Added print
            img = image.load_img(image_path, target_size=target_size)
            print("Debug: Image loaded.") # <-- Added print

            print("Debug: Converting to array and expanding dimensions...") # <-- Added print
            img_array = image.img_to_array(img)
            img_array = np.expand_dims(img_array, axis=0)
            img_array /= 255.
            print(f"Debug: Array shape after processing: {img_array.shape}") # <-- Added print

            print("Debug: Making prediction...") # <-- Added print
            prediction = model.predict(img_array)
            print(f"Debug: Raw prediction output: {prediction}") # <-- Added print

            if prediction[0][0] > 0.5:
                return f"Predicted: Dog ({prediction[0][0]:.4f})"
            else:
                return f"Predicted: Cat ({prediction[0][0]:.4f})"
        except Exception as e:
            print(f"Debug: An exception occurred: {e}") # <-- Added print
            return f"Error processing image {image_path}: {e}"

    # --- Example Usage ---

    # !!! IMPORTANT: Replace these paths with actual paths to NEW images you want to test !!!
    # Use real paths for test_image_path_cat and test_image_path_dog
    test_image_path_cat = '/home/stemland/dataset/test/cat/cat_5.jpg' # <-- REPLACE THIS
    test_image_path_dog = '/home/stemland/dataset/test/dog/dog_68.jpg' # <-- REPLACE THIS
    test_image_path_missing = '/this/path/does/not/exist.jpg' # Keep this as a test for the error handling

    print("\n--- Running Predictions ---") # <-- Added print
    print(predict_cat_or_dog(test_image_path_cat, model))
    print(predict_cat_or_dog(test_image_path_dog, model))
    print(predict_cat_or_dog(test_image_path_missing, model))
    print("--- Predictions Finished ---") # <-- Added print


else:
    print(f"Error: Model file not found at {model_path}.")
    print("Please ensure the model training and saving step was completed successfully in the previous cell.")

2025-05-18 09:53:31.995819: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-18 09:53:32.019504: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-18 09:53:32.204792: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-18 09:53:32.345992: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747542212.490418   15247 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747542212.53

Loading model from cat_dog_model.h5...


2025-05-18 09:53:36.769063: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model loaded successfully.

--- Running Predictions ---

Attempting to predict on image: /home/stemland/dataset/test/cat/cat_5.jpg
Debug: Loading image...
Debug: Image loaded.
Debug: Converting to array and expanding dimensions...
Debug: Array shape after processing: (1, 150, 150, 3)
Debug: Making prediction...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 192ms/step
Debug: Raw prediction output: [[0.5315455]]
Predicted: Dog (0.5315)

Attempting to predict on image: /home/stemland/dataset/test/dog/dog_68.jpg
Debug: Loading image...
Debug: Image loaded.
Debug: Converting to array and expanding dimensions...
Debug: Array shape after processing: (1, 150, 150, 3)
Debug: Making prediction...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
Debug: Raw prediction output: [[0.55218816]]
Predicted: Dog (0.5522)

Attempting to predict on image: /this/path/does/not/exist.jpg
Debug: os.path.exists(/this/path/does/not/exist.jpg) is False
Error: Image not found at /this/path/does/not/exist.jpg
--- Predictions Finished ---


In [3]:
# This is the cell for prediction.
# Run this AFTER the training cell has completed and saved the model.

import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import numpy as np
import os
import sys # Import sys to force flush output

# Load the saved model
model_path = "cat_dog_model.h5"

# Added check to be sure the model file exists before trying to load
if os.path.exists(model_path):
    print(f"Loading model from {model_path}...")
    model = load_model(model_path)
    print("Model loaded successfully.")
    sys.stdout.flush() # Force output immediately

    # Function to predict on a single image
    def predict_cat_or_dog(image_path, model, target_size=(150, 150)):
        print(f"\nAttempting to predict on image: {image_path}")
        sys.stdout.flush() # Force output immediately

        if not os.path.exists(image_path):
            print(f"Debug: os.path.exists({image_path}) is False")
            sys.stdout.flush() # Force output immediately
            return f"Error: Image not found at {image_path}"

        try:
            print("Debug: Image path exists. Proceeding to load...")
            sys.stdout.flush() # Force output immediately

            # Test if the path is actually a file
            if not os.path.isfile(image_path):
                print(f"Debug: Path exists but is not a file: {image_path}")
                sys.stdout.flush() # Force output immediately
                return f"Error: Path is not a file: {image_path}"

            # Test file size
            file_size = os.path.getsize(image_path)
            print(f"Debug: File size is {file_size} bytes.")
            sys.stdout.flush() # Force output immediately
            if file_size == 0:
                 print(f"Debug: File is empty: {image_path}")
                 sys.stdout.flush() # Force output immediately
                 return f"Error: Image file is empty: {image_path}"


            print("Debug: Loading image with image.load_img...")
            sys.stdout.flush() # Force output immediately
            img = image.load_img(image_path, target_size=target_size)
            print("Debug: image.load_img completed.")
            sys.stdout.flush() # Force output immediately


            print("Debug: Converting to array and expanding dimensions...")
            sys.stdout.flush() # Force output immediately
            img_array = image.img_to_array(img)
            img_array = np.expand_dims(img_array, axis=0)
            img_array /= 255.
            print(f"Debug: Array shape after processing: {img_array.shape}")
            sys.stdout.flush() # Force output immediately

            print("Debug: Making prediction...")
            sys.stdout.flush() # Force output immediately
            prediction = model.predict(img_array)
            print(f"Debug: Raw prediction output: {prediction}")
            sys.stdout.flush() # Force output immediately

            if prediction[0][0] > 0.5:
                return f"Predicted: Dog ({prediction[0][0]:.4f})"
            else:
                return f"Predicted: Cat ({prediction[0][0]:.4f})"
        except Exception as e:
            # Print the full traceback for better debugging
            import traceback
            traceback.print_exc()
            print(f"Debug: An exception occurred: {e}")
            sys.stdout.flush() # Force output immediately
            return f"Error processing image {image_path}: {e}"

    # --- Example Usage ---

    # !!! IMPORTANT: Replace these paths with actual paths to NEW images you want to test !!!
    # Use real paths for test_image_path_cat and test_image_path_dog
    test_image_path_cat = '/home/stemland/dataset/test/cat/cat_88.jpg' # <-- REPLACE THIS
    test_image_path_dog = '/home/stemland/dataset/test/dog/dog_520.jpg' # <-- REPLACE THIS
    test_image_path_missing = '/this/path/does/not/exis.jpg' # Keep this as a test for the error handling

    print("\n--- Running Predictions ---")
    sys.stdout.flush() # Force output immediately

    # Test one image at a time to isolate issues
    print(predict_cat_or_dog(test_image_path_dog, model))
    # print(predict_cat_or_dog(test_image_path_dog, model)) # Comment out for initial test
    # print(predict_cat_or_dog(test_image_path_missing, model)) # Comment out for initial test

    print("--- Predictions Finished ---")
    sys.stdout.flush() # Force output immediately


else:
    print(f"Error: Model file not found at {model_path}.")
    print("Please ensure the model training and saving step was completed successfully in the previous cell.")
    sys.stdout.flush() # Force output immediately

Loading model from cat_dog_model.h5...


Model loaded successfully.

--- Running Predictions ---

Attempting to predict on image: /home/stemland/dataset/test/dog/dog_520.jpg
Debug: Image path exists. Proceeding to load...
Debug: File size is 31454 bytes.
Debug: Loading image with image.load_img...
Debug: image.load_img completed.
Debug: Converting to array and expanding dimensions...
Debug: Array shape after processing: (1, 150, 150, 3)
Debug: Making prediction...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step
Debug: Raw prediction output: [[0.5617407]]
Predicted: Dog (0.5617)
--- Predictions Finished ---


In [2]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

# Paths to your folders
train_dir = '/home/stemland/dataset/train'
test_dir = '/home/stemland/dataset/test'

# Preprocess images: rescale and optionally split validation
train_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# Load training and testing images
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary'  # cat=0, dog=1
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary'
)

# Build a simple CNN model
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(128, 128, 3)),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')  # For binary classification
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(train_generator, epochs=5)

# Evaluate on test data
loss, accuracy = model.evaluate(test_generator)
print(f"Test Accuracy: {accuracy * 100:.2f}%")


Found 1112 images belonging to 2 classes.
Found 428 images belonging to 2 classes.
Epoch 1/5
35/35 ━━━━━━━━━━━━━━━━━━━━ 13s 350ms/step - accuracy: 0.5766 - loss: 1.5705
Epoch 2/5
35/35 ━━━━━━━━━━━━━━━━━━━━ 13s 381ms/step - accuracy: 0.8434 - loss: 0.3790
Epoch 3/5
35/35 ━━━━━━━━━━━━━━━━━━━━ 15s 435ms/step - accuracy: 0.9313 - loss: 0.2089
Epoch 4/5
35/35 ━━━━━━━━━━━━━━━━━━━━ 12s 352ms/step - accuracy: 0.9911 - loss: 0.0721
Epoch 5/5
35/35 ━━━━━━━━━━━━━━━━━━━━ 14s 399ms/step - accuracy: 0.9976 - loss: 0.0307
14/14 ━━━━━━━━━━━━━━━━━━━━ 4s 272ms/step - accuracy: 0.9596 - loss: 0.0979
Test Accuracy: 96.26%


In [3]:
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import numpy as np
import os
import sys

model_path = "cat_dog_model.h5"

if os.path.exists(model_path):
    print(f"Loading model from {model_path}...")
    model = load_model(model_path)
    print("Model loaded successfully.")
    sys.stdout.flush()

    def predict_cat_or_dog(image_path, model, target_size=(150, 150)):
        print(f"\nPredicting image: {image_path}")
        sys.stdout.flush()

        if not os.path.exists(image_path):
            return f"Error: Image not found at {image_path}"

        try:
            if not os.path.isfile(image_path):
                return f"Error: Path is not a file: {image_path}"

            if os.path.getsize(image_path) == 0:
                return f"Error: Image file is empty: {image_path}"

            img = image.load_img(image_path, target_size=target_size)
            img_array = image.img_to_array(img)
            img_array = np.expand_dims(img_array, axis=0) / 255.0

            prediction = model.predict(img_array)[0][0]

            if prediction > 0.5:
                return f"Predicted: Cat (confidence: {prediction:.4f})"
            else:
                return f"Predicted: Dog (confidence: {1 - prediction:.4f})"

        except Exception as e:
            return f"Error processing image {image_path}: {e}"

    # Example usage — replace with your actual image paths:
    test_images = [
        '/home/stemland/dataset/test/cat/cat_56.jpg',
        '/home/stemland/dataset/test/dog/00824-3846168986.png',
        '/home/stemland/dataset/test/cat/00328-200124638.png'  # New PNG image example
    ]

    for img_path in test_images:
        print(predict_cat_or_dog(img_path, model))

else:
    print(f"Error: Model file not found at {model_path}. Please train and save the model first.")


Loading model from cat_dog_model.h5...
Model loaded successfully.

Predicting image: /home/stemland/dataset/test/cat/cat_56.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step
Predicted: Cat (confidence: 0.8400)

Predicting image: /home/stemland/dataset/test/dog/00824-3846168986.png
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
Predicted: Cat (confidence: 0.8188)

Predicting image: /home/stemland/dataset/test/cat/00328-200124638.png
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
Predicted: Cat (confidence: 0.9252)


In [5]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Paths
train_dir = '/home/stemland/dataset/train'
test_dir = '/home/stemland/dataset/test'

# Data Augmentation for training data
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.15,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2  # 20% for validation
)

test_datagen = ImageDataGenerator(rescale=1./255)

# Generators
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(128,128),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

validation_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(128,128),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(128,128),
    batch_size=32,
    class_mode='binary'
)

# Improved CNN Model
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Early stopping to stop training when val loss doesn't improve for 5 epochs
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(
    train_generator,
    epochs=30,
    validation_data=validation_generator,
    callbacks=[early_stopping]
)

# Evaluate on test data
loss, accuracy = model.evaluate(test_generator)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

# Save the improved model
model.save("cat_dog_model_improved.h5")


Found 890 images belonging to 2 classes.
Found 222 images belonging to 2 classes.
Found 428 images belonging to 2 classes.
Epoch 1/30
28/28 ━━━━━━━━━━━━━━━━━━━━ 17s 578ms/step - accuracy: 0.5397 - loss: 0.7020 - val_accuracy: 0.7162 - val_loss: 0.6622
Epoch 2/30
28/28 ━━━━━━━━━━━━━━━━━━━━ 16s 574ms/step - accuracy: 0.6590 - loss: 0.6407 - val_accuracy: 0.7162 - val_loss: 0.6200
Epoch 3/30
28/28 ━━━━━━━━━━━━━━━━━━━━ 15s 545ms/step - accuracy: 0.6885 - loss: 0.6129 - val_accuracy: 0.6441 - val_loss: 0.6047
Epoch 4/30
28/28 ━━━━━━━━━━━━━━━━━━━━ 16s 558ms/step - accuracy: 0.6967 - loss: 0.5855 - val_accuracy: 0.8378 - val_loss: 0.4235
Epoch 5/30
28/28 ━━━━━━━━━━━━━━━━━━━━ 15s 541ms/step - accuracy: 0.7989 - loss: 0.4474 - val_accuracy: 0.9459 - val_loss: 0.2404
Epoch 6/30
28/28 ━━━━━━━━━━━━━━━━━━━━ 15s 533ms/step - accuracy: 0.8275 - loss: 0.4024 - val_accuracy: 0.9234 - val_loss: 0.2704
Epoch 7/30
28/28 ━━━━━━━━━━━━━━━━━━━━ 16s 555ms/step - accuracy: 0.8798 - loss: 0.3495 - val_accuracy: 

Test Accuracy: 96.26%


In [9]:
from tensorflow.keras.preprocessing import image
import numpy as np
from tensorflow.keras.models import load_model
import os

model = load_model('cat_dog_model_improved.h5')
print("Model loaded successfully.")

def predict_cat_or_dog(image_path, model, target_size=(224, 224)):
    if not os.path.exists(image_path):
        return f"Image not found: {image_path}"

    img = image.load_img(image_path, target_size=target_size)
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) / 255.0

    prediction = model.predict(img_array)[0][0]
    label = "Dog" if prediction > 0.5 else "Cat"
    confidence = prediction if prediction > 0.5 else 1 - prediction
    return f"Predicted: {label} (confidence: {confidence:.4f})"

# Test
images = [
    '/home/stemland/dataset/test/cat/cat_56.jpg',
    '/home/stemland/dataset/test/dog/00824-3846168986.png',
    '/home/stemland/dataset/test/cat/00328-200124638.png'
]

for img in images:
    print(predict_cat_or_dog(img, model))


Model loaded successfully.


ValueError: Exception encountered when calling Sequential.call().

[1mInput 0 of layer "dense_4" is incompatible with the layer: expected axis -1 of input shape to have value 25088, but received input with shape (1, 86528)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(1, 224, 224, 3), dtype=float32)
  • training=False
  • mask=None

In [10]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator

model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    MaxPooling2D(2, 2),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train using train_generator and val_generator as before
model.fit(train_generator, validation_data=val_generator, epochs=10)

model.save('cat_dog_model_fixed.h5')


NameError: name 'val_generator' is not defined

In [11]:
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import numpy as np
import os
import sys

model_path = "cat_dog_model.h5"

def predict_cat_or_dog_flat(image_path, model, target_size=(150, 150)):
    print(f"\nPredicting image: {image_path}")
    sys.stdout.flush()

    if not os.path.exists(image_path):
        return f"Error: Image not found at {image_path}"

    try:
        if not os.path.isfile(image_path):
            return f"Error: Path is not a file: {image_path}"

        if os.path.getsize(image_path) == 0:
            return f"Error: Image file is empty: {image_path}"

        img = image.load_img(image_path, target_size=target_size)
        img_array = image.img_to_array(img) / 255.0
        img_flat = img_array.flatten()

        # Ensure it matches the expected input shape
        expected_shape = model.input_shape[1]
        if img_flat.shape[0] != expected_shape:
            return f"Error: Model expects input shape {expected_shape}, but got {img_flat.shape[0]}"

        img_input = np.expand_dims(img_flat, axis=0)
        prediction = model.predict(img_input)[0][0]

        if prediction > 0.5:
            return f"Predicted: Dog (confidence: {prediction:.4f})"
        else:
            return f"Predicted: Cat (confidence: {1 - prediction:.4f})"

    except Exception as e:
        return f"Error processing image {image_path}: {e}"

# Main logic
if os.path.exists(model_path):
    print(f"Loading model from {model_path}...")
    model = load_model(model_path)
    print("Model loaded successfully.")
    sys.stdout.flush()

    test_images = [
        '/home/stemland/dataset/test/cat/cat_56.jpg',
        '/home/stemland/dataset/test/dog/00824-3846168986.png',
        '/home/stemland/dataset/test/cat/00328-200124638.png'
    ]

    for img_path in test_images:
        print(predict_cat_or_dog_flat(img_path, model))
else:
    print(f"Error: Model file not found at {model_path}. Please train and save the model first.")


Loading model from cat_dog_model.h5...
Model loaded successfully.

Predicting image: /home/stemland/dataset/test/cat/cat_56.jpg
Error: Model expects input shape 150, but got 67500

Predicting image: /home/stemland/dataset/test/dog/00824-3846168986.png
Error: Model expects input shape 150, but got 67500

Predicting image: /home/stemland/dataset/test/cat/00328-200124638.png
Error: Model expects input shape 150, but got 67500


In [2]:
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import numpy as np
import os

# Load the fixed model
model_path = "cat_dog_model_fixed.h5"
model = load_model(model_path)
print("✅ Model loaded successfully.")

# Prediction function
def predict_cat_or_dog(image_path, model, target_size=(150, 150)):
    print(f"\n📷 Predicting image: {image_path}")
    
    if not os.path.exists(image_path) or os.path.getsize(image_path) == 0:
        return f"❌ Error: Invalid image file: {image_path}"

    try:
        img = image.load_img(image_path, target_size=target_size)
        img_array = image.img_to_array(img) / 255.0
        img_array = np.expand_dims(img_array, axis=0)
        prediction = model.predict(img_array)[0][0]

        if prediction > 0.5:
            return f"🦮 Predicted: Dog (confidence: {prediction:.4f})"
        else:
            return f"🐱 Predicted: Cat (confidence: {1 - prediction:.4f})"
    except Exception as e:
        return f"⚠️ Error: {e}"

# Example usage
test_images = [
    '/home/stemland/dataset/test/cat/cat_56.jpg',
    '/home/stemland/dataset/test/dog/00824-3846168986.png',
    '/home/stemland/dataset/test/cat/00328-200124638.png'
]

for img_path in test_images:
    print(predict_cat_or_dog(img_path, model))


FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'cat_dog_model_fixed.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [4]:
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import numpy as np
import os
import sys

model_path = "cat_dog_model.h5"

if os.path.exists(model_path):
    print(f"Loading model from {model_path}...")
    model = load_model(model_path)
    print("Model loaded successfully.")
    sys.stdout.flush()

    # You must set this based on how you trained the model
    # Adjust manually if you don't have class_indices
    class_indices = {'cat': 0, 'dog': 1}  # OR {'dog': 0, 'cat': 1}
    index_to_label = {v: k.capitalize() for k, v in class_indices.items()}

    def predict_cat_or_dog(image_path, model, target_size=(150, 150)):
        print(f"\nPredicting image: {image_path}")
        sys.stdout.flush()

        if not os.path.exists(image_path):
            return f"Error: Image not found at {image_path}"

        try:
            if not os.path.isfile(image_path):
                return f"Error: Path is not a file: {image_path}"

            if os.path.getsize(image_path) == 0:
                return f"Error: Image file is empty: {image_path}"

            img = image.load_img(image_path, target_size=target_size)
            img_array = image.img_to_array(img)
            img_array = np.expand_dims(img_array, axis=0) / 255.0

            prediction = model.predict(img_array)[0][0]

            # Decide label based on class_indices
            if class_indices['cat'] < class_indices['dog']:
                label = "Dog" if prediction > 0.5 else "Cat"
                confidence = prediction if prediction > 0.5 else 1 - prediction
            else:
                label = "Cat" if prediction > 0.5 else "Dog"
                confidence = prediction if prediction > 0.5 else 1 - prediction

            return f"Predicted: {label} (confidence: {confidence:.4f})"

        except Exception as e:
            return f"Error processing image {image_path}: {e}"

    # Test
    test_images = [
        '/home/stemland/dataset/test/cat/cat_56.jpg',
        '/home/stemland/dataset/test/dog/00824-3846168986.png',
        '/home/stemland/dataset/test/cat/00328-200124638.png'
    ]

    for img_path in test_images:
        print(predict_cat_or_dog(img_path, model))

else:
    print(f"Error: Model file not found at {model_path}. Please train and save the model first.")


Loading model from cat_dog_model.h5...
Model loaded successfully.

Predicting image: /home/stemland/dataset/test/cat/cat_56.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
Predicted: Dog (confidence: 0.8400)

Predicting image: /home/stemland/dataset/test/dog/00824-3846168986.png
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
Predicted: Dog (confidence: 0.8188)

Predicting image: /home/stemland/dataset/test/cat/00328-200124638.png
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
Predicted: Dog (confidence: 0.9252)
